# 📓 Lab 1: Bond Mathematics & Compounding

This notebook contains the complete Python implementations for Lecture 1 (Bond Math) concepts.

## 🕒 Lesson 1.1: Time Value of Money & Compounding

Let's write helper functions to calculate Future Value ($FV$) and Present Value ($PV$) using discrete compounding (compounded $m$ times a year) and continuous compounding.

In [ ]:
import numpy as np

def future_value_discrete(pv, r, n, m=1):
    return pv * (1 + r / m) ** (m * n)

def future_value_continuous(pv, r, n):
    return pv * np.exp(r * n)

def present_value_discrete(fv, r, n, m=1):
    return fv / ((1 + r / m) ** (m * n))

def present_value_continuous(fv, r, n):
    return fv * np.exp(-r * n)

### 📝 Practice Problem Solver:
**Question:** Calculate the money needed to invest today ($PV$) to get $30,000 in 3 years at a 5% interest rate continuously compounded.

In [ ]:
target_fv = 30000
rate = 0.05
years = 3

required_pv = present_value_continuous(target_fv, rate, years)
print(f"To get ${target_fv:,.2f} in {years} years at {rate*100}% continuous compounding, ")
print(f"you need to invest: ${required_pv:,.2f} today.")

## 🎟️ Lesson 1.2: Bond Pricing Mechanics & YTM Solver

In [ ]:
from scipy.optimize import brentq

def price_coupon_bond(face_value, coupon_rate, ytm, t, m=2):
    """
    Calculate the price of a coupon bond.
    m=2 is standard semi-annual compounding.
    """
    n = int(t * m)
    coupon = (coupon_rate * face_value) / m
    periods = np.arange(1, n + 1)
    discount_factors = (1 + ytm / m) ** (-periods)
    price = np.sum(coupon * discount_factors) + face_value * discount_factors[-1]
    return price

def solve_ytm(market_price, face_value, coupon_rate, t, m=2):
    """
    Solve for the Yield to Maturity (YTM) of a coupon bond given its market price.
    """
    # We want to solve for y such that price_coupon_bond(...) - market_price = 0
    objective_function = lambda y: price_coupon_bond(face_value, coupon_rate, y, t, m) - market_price
    
    # Brent's method to find roots in the interval [-20%, 100%]
    return brentq(objective_function, -0.2, 1.0)

# Test Bond Pricing
test_face = 1000
test_coupon = 0.06 # 6% coupon rate
test_years = 5
target_ytm = 0.05 # 5% YTM

p = price_coupon_bond(test_face, test_coupon, target_ytm, test_years)
solved = solve_ytm(p, test_face, test_coupon, test_years)
print(f"Bond Price (at 5% YTM): ${p:.2f} (Should be premium since Coupon Rate > YTM)")
print(f"Solved YTM: {solved * 100:.4f}%")

## 📈 Lesson 1.3: Yield Curves, Spot Rates & Bootstrapping

Let's write a simple bootstrapping function to extract the spot curve from a set of coupon-bearing bonds.

In [ ]:
def bootstrap_spot_rates(bonds):
    """
    bonds is a list of dictionaries with:
    - 't': maturity in years
    - 'price': current market price
    - 'coupon_rate': annual coupon rate (paid annually for simplicity, m=1)
    - 'face_value': face value
    Assumed sorted by maturity 't'.
    """
    spot_rates = {}
    discount_factors = {}
    
    for bond in bonds:
        t = bond['t']
        price = bond['price']
        coupon = bond['coupon_rate'] * bond['face_value']
        face = bond['face_value']
        
        # For coupon payments at previous periods, discount them using known discount factors
        sum_discounted_prev_coupons = 0.0
        for prev_t in range(1, int(t)):
            sum_discounted_prev_coupons += coupon * discount_factors[prev_t]
            
        # The remaining price is for the final payment (coupon + face value) at maturity t
        final_payment = coupon + face
        df_t = (price - sum_discounted_prev_coupons) / final_payment
        
        discount_factors[t] = df_t
        # Continuously compounded spot rate: df(t) = exp(-r * t) => r = -ln(df(t)) / t
        spot_rates[t] = -np.log(df_t) / t
        
    return spot_rates, discount_factors

# Sample market data: 1-year, 2-year, 3-year bonds
market_bonds = [
    {'t': 1, 'price': 98.0, 'coupon_rate': 0.03, 'face_value': 100},
    {'t': 2, 'price': 99.0, 'coupon_rate': 0.04, 'face_value': 100},
    {'t': 3, 'price': 101.0, 'coupon_rate': 0.05, 'face_value': 100}
]

spots, dfs = bootstrap_spot_rates(market_bonds)
print("Bootstrapped Spot Rates (Continuous):")
for t, rate in spots.items():
    print(f"  {t}-Year Spot Rate: {rate*100:.4f}% (Discount Factor: {dfs[t]:.4f})")

## ⚖️ Lessons 1.4 & 1.5: Risk Sensitivity (Duration & Convexity)

Let's compute Macaulay Duration, Modified Duration, and Convexity for a bond, and verify the Taylor approximation for a yield shock.

In [ ]:
def bond_duration_and_convexity(face_value, coupon_rate, ytm, t, m=2):
    """
    Calculates price, Macaulay Duration, Modified Duration, and Convexity (Discrete terms).
    """
    n = int(t * m)
    coupon = (coupon_rate * face_value) / m
    periods = np.arange(1, n + 1)
    times = periods / m
    
    # Cash flows
    cfs = np.full(n, coupon)
    cfs[-1] += face_value
    
    # Present values of cash flows
    pv_cfs = cfs / ((1 + ytm / m) ** periods)
    price = np.sum(pv_cfs)
    
    # Macaulay Duration
    mac_duration = np.sum(times * pv_cfs) / price
    
    # Modified Duration
    mod_duration = mac_duration / (1 + ytm / m)
    
    # Convexity (Discrete compounding definition)
    convexity = np.sum(periods * (periods + 1) * pv_cfs) / (price * m**2 * (1 + ytm / m)**2)
    
    return price, mac_duration, mod_duration, convexity

# Set up a 10-year 6% bond trading at 5% YTM
p, mac_d, mod_d, conv = bond_duration_and_convexity(1000, 0.06, 0.05, 10, m=2)
print(f"Bond Price: ${p:.2f}")
print(f"Macaulay Duration: {mac_d:.4f} Years")
print(f"Modified Duration: {mod_d:.4f}")
print(f"Convexity: {conv:.4f}")

print("\n--- Verify Taylor Series Approximation ---")
dy = 0.01 # +100 bps shift
price_up = price_coupon_bond(1000, 0.06, 0.05 + dy, 10, m=2)
actual_pct_change = (price_up - p) / p

duration_only_pct = -mod_d * dy
duration_and_conv_pct = -mod_d * dy + 0.5 * conv * dy**2

print(f"Actual % Price Change for +100bps: {actual_pct_change * 100:.4f}%")
print(f"Duration-Only Estimate:            {duration_only_pct * 100:.4f}% (Error: {(duration_only_pct - actual_pct_change)*100:.4f}%) ")
print(f"Duration + Convexity Estimate:     {duration_and_conv_pct * 100:.4f}% (Error: {(duration_and_conv_pct - actual_pct_change)*100:.4f}%)")